## Спринт 1.
Задачи:
- Найти по крайней мере 5 современных оптимизаторов, заполнить информацию о них: название, дата выхода, гиперпараметры, идея, список статей по ним
- Посмотреть зарубежные исследования по применению оптимизаторов, посмотреть на датасеты, которые там используются (отчет)
- Обучение простой нейронной сети для задачи регрессии (желательно использовать датасеты из исследований) (notebook)


In [2]:
%pip install -qUr requirements.txt


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Импортируем необходимые модули.  

In [16]:
from datetime import datetime

import os

from typing import Optional, Dict, Tuple, List

import numpy as np

import pandas as pd

import torch
from torch import nn
import torch.utils.tensorboard
from torch.utils.tensorboard.writer import SummaryWriter
from torch.optim import AdamW
from torchmetrics.regression import MeanAbsolutePercentageError as MAPE

from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler

from adabelief_pytorch import AdaBelief
from lion_pytorch import Lion

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else device)
device

Введем функцию `preprocess_data`, которая читает данные в формате `csv` и преобрабатывает их.  

In [5]:
def preprocess_data(filename: str) -> torch.Tensor:
    """Function to preprocess data

    Args:
        filename (str): filename of data to be preprocessed

    Returns:
        torch.Tensor: preprocessed data
    """
    data = pd.read_csv(filename)
    
    data_t = torch.Tensor(data.values)
    data_t = torch.nn.functional.normalize(data_t)
    
    return data_t


Введем функцию `create_writer`, возвращающую объект класса `SummaryWriter` для записи всех экспериментов с разными параметрами.  

In [6]:
def create_writer(
    experiment_name: str,
    model_name: str,
    extra: Optional[str] = None,
) -> torch.utils.tensorboard.writer.SummaryWriter:
    """Creates a torch.utils.tensorboard.writer.SummaryWriter instance saving to log_dir.
    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.
    timestamp at format YYYY-MM-DD.
    Args:
        experiment_name (str): name of experiment
        model_name (str): name of model
        extra (Optional[str]): Anything extra to add to the directory. Defaults to None.

    Returns:
        SummaryWriter: _description_
    """
    timestamp = datetime.now().strftime("%Y-%m-%d")
    if extra:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)

    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

Введем 3 функции: `train` - в ней реализован цикл обучения модели, `train_step` и `test_step` - функции одной итерации для обучения модели и ее валидации.  

In [ ]:
def train_step(
    model: nn.Module,
    train_data: torch.Tensor,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Args:
        model (nn.Module): A PyTorch model to be trained.
        train_data (torch.Tensor): A tensor data that model is trained on.
        loss_fn (nn.Module): A PyTorch loss function to minimize.
        optimizer (torch.optim.Optimizer): A PyTorch optimizer to help minimize the loss function.
        device (torch.device): A target device to compute on (e.g. "cuda" or "cpu").

        Returns:
        A Tuple[float, float] of training metrics.
    """
    model.train()

    X, y = train_data[:, :-1].to(device), train_data[:, -1].to(device)
    y = y.reshape(-1, 1)
    y_pred = model(X)

    loss = loss_fn(y_pred, y)
    train_loss = loss.item()
    train_mape = MAPE().to(device)(y_pred, y).item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return train_loss, train_mape

In [ ]:
def test_step(
    model: nn.Module,
    test_data: torch.Tensor,
    loss_fn: nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
        model (nn.Module): A PyTorch model to be tested.
        test_data (torch.Tensor): A tensor data that model is tested on.
        loss_fn (nn.Module): A PyTorch loss function to calculate loss on the test data.
        device (torch.device): A target device to compute on ("cuda", "cpu", "mps").

    Returns:
        A Tuple[float, float] of testing metrics.
    """
    model.eval()

    with torch.inference_mode():
        X, y = test_data[:, :-1].to(device), test_data[:, -1].to(device)
        y = y.reshape(-1, 1)
        test_pred = model(X)

        loss = loss_fn(test_pred, y)
        test_loss = loss.item()
        test_mape = MAPE().to(device)(test_pred, y).item()

    return test_loss, test_mape

In [11]:
def train(
    model: torch.nn.Module,
    train_data: torch.Tensor,
    test_data: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    device: torch.device,
    scheduler: Optional[torch.optim.lr_scheduler.ReduceLROnPlateau] = None,
    writer: Optional[SummaryWriter] = None,
    epoch_step: int = 10,
) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
        model: A PyTorch model to be trained and tested.
        train_data: A data for the model to be trained on.
        test_data: A data for the model to be tested on.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        loss_fn: A PyTorch loss function to calculate loss on both datasets.
        epochs: An integer indicating how many epochs to train for.
        device: A target device to compute on (e.g. "cuda", "cpu", "mps").
        scheduler: A LRScheduler intance for interrupting training.
        writer: A SummaryWriter intance for writing experiment results.
        epoch_step: A number to represent step count to output information about training.

    Returns:
        A dictionary of training and testing loss as well as training and
        testing MAPE metrics. Each metric has a value in a list for
        each epoch.
    """
    results: Dict[str, List] = {
        "train_loss": [],
        "test_loss": [],
        "train_mape": [],
        "test_mape": [],
    }

    for epoch in range(epochs):
        train_loss, train_mape = train_step(
            model=model,
            train_data=train_data,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
        )

        if scheduler is not None:
            scheduler.step(train_loss)

        test_loss, test_mape = test_step(
            model=model,
            test_data=test_data,
            loss_fn=loss_fn,
            device=device,
        )
        if epoch % epoch_step == 0:
            # print(
            #    f"Epoch: {epoch} | "
            #    f"train_loss: {train_loss:.4f} | "
            #    f"test_loss: {test_loss:.4f} | "
            #    f"train_mape: {train_mape:.4f} | "
            #    f"test_mape: {test_mape:.4f} | "
            # )

            results["train_loss"].append(train_loss)
            results["test_loss"].append(test_loss)
            results["train_mape"].append(train_mape)
            results["test_mape"].append(test_mape)

            if writer is not None:
                writer.add_scalars(
                    main_tag="Loss",
                    tag_scalar_dict={
                        "train_loss": train_loss,
                        "test_loss": test_loss,
                        "train_mape": train_mape,
                        "test_mape": test_mape,
                    },
                    global_step=epoch,
                )

                writer.close()
            else:
                pass

    return results

Создадим бейcлайн `SimpleRegressionModel`.  

In [20]:
class SimpleRegressionModel(nn.Module):
    def __init__(
        self,
        input_shape: int = 8,
        hidden_units: int = 16,
        output_shape: int = 1,
        activation_function: Optional[torch.nn.Module] = None,
        batch_normalization: Optional[torch.nn.Module] = None,
        dropout: Optional[torch.nn.Module] = None,
    ) -> None:
        """SimpleRegressionModel initializer

        Args:
            input_shape (int): Number of units in input layer
            hidden_units (int): Number of units in hidden layers
            output_shape (int): Number of units in output layer
            activation_function (Optional[torch.nn.Module], optional): Activation function. Defaults to None.
            batch_normalization (Optional[torch.nn.Module], optional): Batch normalization layer. Defaults to None.
            dropout (Optional[torch.nn.Module], optional): Dropout layer. Defaults to None.
        """
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(
                in_features=input_shape,
                out_features=hidden_units,
                device=device,
            ),
            nn.Identity() if batch_normalization is None else batch_normalization,
            nn.Identity() if activation_function is None else activation_function,
            nn.Identity() if dropout is None else dropout,
            nn.Linear(
                in_features=hidden_units,
                out_features=output_shape,
                device=device,
            ),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """Method for forward pass.

        Args:
            X (torch.Tensor): input data

        Returns:
            torch.Tensor: result of computations
        """
        return self.block(X)

## Спринт 2.  
- Ознакомиться с подходами BatchNorm, LayerNorm, Dropout. Внедрить их в текущую сеть. Изучить виды функций активации, чем они отличаются и протестировать изменение точности модели при смене функции активации
- Попробовать на практике применить алгоритмы AdaBelief, AdamW (пока базовые варианты без подбора параметров), посмотреть что происходит с точностью модели, сравнить с результатами LION


Зададим словари параметров модели и оптимизаторов, сделаем их подбор с использованием библиотеки `ray`.  

In [ ]:
INPUT_SHAPE = 8
HIDDEN_UNITS = 16
OUTPUT_SHAPE = 1
NUM_EPOCHS = 31
EPOCH_STEP = 5
LEARNING_RATE = 0.001
train_filename = "data/california_housing_test.csv"
test_filename = "data/california_housing_train.csv"
np.random.seed(0)
torch.manual_seed(0)


model_params = {
    "activation_function": tune.choice(
        [None, nn.ReLU(), nn.Sigmoid(), nn.Tanh()]
    ),  # tune.grid_search
    "batch_normalization": tune.choice(
        [None, nn.BatchNorm1d(num_features=HIDDEN_UNITS)]
    ),
    "dropout": tune.choice(
        [
            None,
            nn.Dropout1d(p=0.2),
            nn.Dropout1d(p=0.3),
            nn.Dropout1d(p=0.5),
        ]
    ),
}

optimizer_params = {
    Lion: {"lr": tune.choice([1e-4, 1e-3, 1e-2])},
    AdaBelief: {
        "lr": tune.choice([1e-4, 1e-3, 1e-2]),
        "print_change_log": tune.choice([False]),
        "rectify": tune.choice([False]),
        "weight_decouple": tune.choice([False]),
    },
    AdamW: {
        "lr": tune.choice([1e-4, 1e-3, 1e-2]),
    },
}

train_data = preprocess_data(train_filename)
test_data = preprocess_data(test_filename)

writer = create_writer(
    experiment_name="regression",
    model_name="simple_reg_model",
)

analysis_optimizer = {}
reporter = CLIReporter(metric_columns=["test_mape"], mode="min")

for optimizer_type in optimizer_params.keys():

    def wrap(config: dict) -> None:
        """Wrapper function for tuning hyperparameters.

        Args:
            config: A dict whose keys are parameters to be tuned and values that the parameter can take.

        """
        model = SimpleRegressionModel(
            input_shape=8,
            hidden_units=16,
            output_shape=1,
            **{i: config[i] for i in config if i in model_params}
        ).to(device)
        loss_fn = torch.nn.MSELoss()
        optimizer = optimizer_type(
            model.parameters(),
            **{i: config[i] for i in config if i in optimizer_params[optimizer_type]}
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=5, factor=0.5
        )
        d = train(
            model=model,
            train_data=train_data,
            test_data=test_data,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=NUM_EPOCHS,
            device=device,
            scheduler=scheduler,
            epoch_step=EPOCH_STEP,
        )
        tune.report({"test_mape": d["test_mape"][-1]})

    analysis = tune.run(
        wrap,
        config=model_params
        | {str(key): value for key, value in optimizer_params[optimizer_type].items()},
        scheduler=ASHAScheduler(metric="test_mape", mode="min"),
        num_samples=100,
        progress_reporter=reporter,
        resources_per_trial={"cpu": 2, "gpu": 1},
    )
    analysis_optimizer[optimizer_type] = analysis

/Users/user/Desktop/baza/optimizers_comparison/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/user/Desktop/baza/optimizers_comparison/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/user/Desktop/baza/optimizers_comparison/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])).

In [ ]:
optimizer_comparison_df = pd.DataFrame(
    {
        "Optimizer": [repr(i)[repr(i).rfind(".") + 1 : -2] for i in analysis_optimizer],
        "Config": [
            i.get_best_config(metric="test_mape", mode="min")
            for i in analysis_optimizer.values()
        ],
        "MAPE": [
            i.get_best_trial(metric="test_mape", mode="min").last_result["test_mape"]
            for i in analysis_optimizer.values()
        ],
        "Time_total_s": [
            i.get_best_trial(metric="test_mape", mode="min").last_result["time_total_s"]
            for i in analysis_optimizer.values()
        ],
    }
)
optimizer_comparison_df